In [2]:
print("Working in the east")

Working in the east


In [6]:
# from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFaceEndpoint
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from youtube_transcript_api import YouTubeTranscriptApi

# Indexing (Document Ingestion)

In [ ]:
video_id = "I8vHUv1gFws"

try:
    transcript_list = YouTubeTranscriptApi.get_transcript(video_id,language="en")
    transcript = " ".join(chunk["text"] for chunk in transcript_list)
    print(transcript)
except TranscriptsDisabled:
    print("No captions available for this video.") 
    
    

AttributeError: type object 'YouTubeTranscriptApi' has no attribute 'get_transcript'

In [11]:
from youtube_transcript_api import YouTubeTranscriptApi

video_id = "I8vHUv1gFws"

try:
    transcript_list = YouTubeTranscriptApi().fetch(
        video_id,
        languages=["en"]
    )

    transcript = " ".join(
        snippet.text for snippet in transcript_list
    )

    print(transcript)

except Exception as e:
    print(f"Error: {e}")

Error: 
Could not retrieve a transcript for the video https://www.youtube.com/watch?v=I8vHUv1gFws! This is most likely caused by:

YouTube is blocking requests from your IP. This usually is due to one of the following reasons:
- You have done too many requests and your IP has been blocked by YouTube
- You are doing requests from an IP belonging to a cloud provider (like AWS, Google Cloud Platform, Azure, etc.). Unfortunately, most IPs from cloud providers are blocked by YouTube.

Ways to work around this are explained in the "Working around IP bans" section of the README (https://github.com/jdepoix/youtube-transcript-api?tab=readme-ov-file#working-around-ip-bans-requestblocked-or-ipblocked-exception).


If you are sure that the described cause is not responsible for this error and that a transcript should be retrievable, please create an issue at https://github.com/jdepoix/youtube-transcript-api/issues. Please add which version of youtube_transcript_api you are using and provide the in

In [9]:
pip show youtube-transcript-api

Name: youtube-transcript-api
Version: 1.2.4
Summary: This is a python API which allows you to get the transcripts/subtitles for a given YouTube video. It also works for automatically generated subtitles, supports translating subtitles and it does not require a headless browser, like other selenium based solutions do!
Home-page: 
Author: Jonas Depoix
Author-email: jonas.depoix@web.de
License: MIT
Location: /mnt/c/Users/vinay kumar/Desktop/LangChain_and_RAG/venv/lib/python3.10/site-packages
Requires: defusedxml, requests
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [13]:
with open("transcript.txt", "r", encoding="utf-8") as file:
    transcript = file.read()

print(transcript)

# Introduction to Retrieval-Augmented Generation (RAG)

Hello everyone. In this video, we are going to learn about Retrieval-Augmented Generation, commonly known as RAG.

RAG is a technique that combines information retrieval with large language models. It helps language models answer questions using external knowledge sources instead of relying only on the information learned during training.

Let's understand why RAG is useful. Large language models can generate human-like responses, but they may sometimes produce incorrect information or hallucinate. They may also lack access to private documents, company data, or recently updated information.

RAG helps address these limitations by retrieving relevant information from an external knowledge base before generating an answer.

A typical RAG pipeline has several important steps.

The first step is document loading. We collect information from sources such as PDF files, text files, websites, or YouTube transcripts.

The second step is t

# Indexing (Splitting the Text)

In [19]:
splitter = RecursiveCharacterTextSplitter(chunk_size=100,chunk_overlap=20)
chunks = splitter.create_documents([transcript])

In [20]:
print(len(chunks))

49


In [29]:
print(chunks)

[Document(metadata={}, page_content='# Introduction to Retrieval-Augmented Generation (RAG)'), Document(metadata={}, page_content='Hello everyone. In this video, we are going to learn about Retrieval-Augmented Generation, commonly'), Document(metadata={}, page_content='commonly known as RAG.'), Document(metadata={}, page_content='RAG is a technique that combines information retrieval with large language models. It helps'), Document(metadata={}, page_content='models. It helps language models answer questions using external knowledge sources instead of'), Document(metadata={}, page_content='sources instead of relying only on the information learned during training.'), Document(metadata={}, page_content="Let's understand why RAG is useful. Large language models can generate human-like responses, but"), Document(metadata={}, page_content='responses, but they may sometimes produce incorrect information or hallucinate. They may also lack'), Document(metadata={}, page_content='They may also l

In [34]:
for i in chunks:
  print(i)

page_content='# Introduction to Retrieval-Augmented Generation (RAG)'
page_content='Hello everyone. In this video, we are going to learn about Retrieval-Augmented Generation, commonly'
page_content='commonly known as RAG.'
page_content='RAG is a technique that combines information retrieval with large language models. It helps'
page_content='models. It helps language models answer questions using external knowledge sources instead of'
page_content='sources instead of relying only on the information learned during training.'
page_content='Let's understand why RAG is useful. Large language models can generate human-like responses, but'
page_content='responses, but they may sometimes produce incorrect information or hallucinate. They may also lack'
page_content='They may also lack access to private documents, company data, or recently updated information.'
page_content='RAG helps address these limitations by retrieving relevant information from an external knowledge'
page_content='externa

# Indexing (Embedding Generation and Storing in Vector Store)

In [ ]:
# Using opensource embedding model "Qwen/Qwen3-Embedding-0.6B"
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="Qwen/Qwen3-Embedding-0.6B",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)





/mnt/c/Users/vinay kumar/Desktop/LangChain_and_RAG/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 310/310 [00:00<00:00, 1470.73it/s]


In [50]:
vector_store = FAISS.from_documents(chunks,embeddings)

In [51]:
vector_store.index_to_docstore_id

{0: '2ed5d3b7-47c7-4ff5-9ed4-1fe30f3fa702',
 1: '0efcd43d-33a2-47b5-ac76-44900ad670ca',
 2: 'f392b72a-3d8e-4677-b264-d05bf1d36ffb',
 3: '76750938-5cef-46a9-bec7-53bbcacc6601',
 4: '146d3402-49e4-4c53-870a-42ddf452cca1',
 5: 'eb101354-1059-4fee-85bd-0d22598ce386',
 6: 'f5de0f5c-f6bb-41d7-8cfc-728fedc23fd9',
 7: '4e8d32a5-109e-43e5-ad4e-3cf8279c1835',
 8: '496f99e1-332b-42d1-ac76-d558cab23c34',
 9: '8d2582f7-9807-4184-b384-0f4c43983f9d',
 10: '53b51c3b-cbbf-4d06-b84c-266481379553',
 11: '487677ed-5213-4abb-8e5c-c3a20915b8e6',
 12: 'c53be276-5a90-4b34-92ba-f24e62d4df34',
 13: '6bd0bbd2-529c-4f70-898e-7e1b7e857372',
 14: '0ad77209-6d77-41c2-bbfb-6bf65631f60c',
 15: 'fae83184-8e8c-4488-b3b3-28141e1b1d32',
 16: '535dc7b2-feff-4f6d-817a-68cbf45edce6',
 17: '5ab7c288-2ec8-4037-b67b-bab404f4379b',
 18: '29fd240a-1456-454e-a62e-58a1fb34dae7',
 19: '10faf452-f5ec-44c7-89e9-14ba6c593468',
 20: '4e7bb062-6966-4949-81ad-c8af87b8530a',
 21: '90558783-3ff0-4ab6-959a-871e9e240fab',
 22: '100c9225-7ff7-

In [53]:
vector_store.get_by_ids(['155991f4-e76e-4ae1-b0db-bd6d4f926915'])

[Document(id='155991f4-e76e-4ae1-b0db-bd6d4f926915', metadata={}, page_content='internal knowledge assistants.')]

# Step2 - Retrivel

In [58]:
retriever = vector_store.as_retriever(search_type="similarity",search_kwargs={"k":4})

In [59]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x78a8f94891b0>, search_kwargs={'k': 4})

In [60]:
retriever.invoke('What is RAG and Why it is used')

[Document(id='2ed5d3b7-47c7-4ff5-9ed4-1fe30f3fa702', metadata={}, page_content='# Introduction to Retrieval-Augmented Generation (RAG)'),
 Document(id='f392b72a-3d8e-4677-b264-d05bf1d36ffb', metadata={}, page_content='commonly known as RAG.'),
 Document(id='dd9dcef8-a203-4cfa-93b9-8eb38402a914', metadata={}, page_content='size, embedding quality, retrieval accuracy, and evaluation are important parts of a good RAG'),
 Document(id='eb0d332e-da32-45f6-859a-272dc9bc8f82', metadata={}, page_content='parts of a good RAG system.')]

## Augmentation

In [62]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage

In [64]:
llm = ChatOllama(
    model="qwen2.5:7b",
    temperature=0.1
)

In [67]:
prompt = PromptTemplate(
  template = """
   You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
  """,
  input_variables=['context','question']
)

In [68]:
question = "Who invented the RAG in the first and why it is used explain me detail"
retrieved_docs = retriever.invoke(question)

In [69]:
retrieved_docs

[Document(id='f392b72a-3d8e-4677-b264-d05bf1d36ffb', metadata={}, page_content='commonly known as RAG.'),
 Document(id='2ed5d3b7-47c7-4ff5-9ed4-1fe30f3fa702', metadata={}, page_content='# Introduction to Retrieval-Augmented Generation (RAG)'),
 Document(id='dd9dcef8-a203-4cfa-93b9-8eb38402a914', metadata={}, page_content='size, embedding quality, retrieval accuracy, and evaluation are important parts of a good RAG'),
 Document(id='eb0d332e-da32-45f6-859a-272dc9bc8f82', metadata={}, page_content='parts of a good RAG system.')]

In [70]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

'commonly known as RAG.\n\n# Introduction to Retrieval-Augmented Generation (RAG)\n\nsize, embedding quality, retrieval accuracy, and evaluation are important parts of a good RAG\n\nparts of a good RAG system.'

In [71]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [72]:
final_prompt

StringPromptValue(text="\n   You are a helpful assistant.\n      Answer ONLY from the provided transcript context.\n      If the context is insufficient, just say you don't know.\n\n      commonly known as RAG.\n\n# Introduction to Retrieval-Augmented Generation (RAG)\n\nsize, embedding quality, retrieval accuracy, and evaluation are important parts of a good RAG\n\nparts of a good RAG system.\n      Question: Who invented the RAG in the first and why it is used explain me detail\n  ")

# Generation

In [73]:
answer = llm.invoke(final_prompt)

In [74]:
print(answer.content)

The context provided does not contain specific information about who invented RAG or the detailed reasons for its use. Therefore, I don't know the exact inventor of RAG or the detailed reasons for its use based solely on this information.


## Building a Chain



In [76]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser


In [77]:
def format_docs(retrieved_docs):
  context_text ="\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [78]:
parallel_chain = RunnableParallel({
  'context' : retriever | RunnableLambda(format_docs),
  'question' : RunnablePassthrough()
})

In [80]:
parallel_chain.invoke('what is RAG')

{'context': '# Introduction to Retrieval-Augmented Generation (RAG)\n\nsize, embedding quality, retrieval accuracy, and evaluation are important parts of a good RAG\n\nparts of a good RAG system.\n\ncommonly known as RAG.',
 'question': 'what is RAG'}

In [81]:
parser = StrOutputParser()

In [82]:
main_chain = parallel_chain | prompt | llm | parser 

In [83]:
main_chain.invoke("Can you summarize the RAG")

"A good RAG (Retrieval-Augmented Generation) system relies on several key components: size, embedding quality, retrieval accuracy, and evaluation. These elements work together to enhance the system's performance in generating relevant and accurate responses."